# RAG Chat with FAISS Index

This notebook implements a simple RAG (Retrieval-Augmented Generation) chat system over a FAISS vector index built from the EmergPhase_EN.pdf document.

## Setup

First, mount your Google Drive and install dependencies.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Dependencies

In [2]:
%pip install langchain langchain-community langchain-openai langchain-text-splitters sentence-transformers faiss-cpu pypdf openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Load Index and Chunks

Load the FAISS vector store, embeddings model, and chunks metadata.

In [3]:
import os
import json
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Paths
DRIVE_ROOT = "/content/drive/MyDrive/iran-prosperity-rag"
INDEX_DIR = f"{DRIVE_ROOT}/data/index"
CHUNKS_JSON = f"{INDEX_DIR}/chunks.json"

# Check if paths exist
if not os.path.exists(INDEX_DIR):
    raise FileNotFoundError(f"Index directory not found: {INDEX_DIR}\nPlease run build_index.py first!")

if not os.path.exists(CHUNKS_JSON):
    raise FileNotFoundError(f"Chunks JSON not found: {CHUNKS_JSON}\nPlease run build_index.py first!")

# Load embeddings model
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load FAISS vector store
print(f"Loading FAISS index from: {INDEX_DIR}")
vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
print("FAISS index loaded successfully!")

# Load chunks metadata
print(f"Loading chunks metadata from: {CHUNKS_JSON}")
with open(CHUNKS_JSON, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
print(f"Loaded {len(chunks_data)} chunks from metadata")

print("\nSetup complete!")

Loading embeddings model...


/tmp/ipython-input-3108842734.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index from: /content/drive/MyDrive/iran-prosperity-rag/data/index
FAISS index loaded successfully!
Loading chunks metadata from: /content/drive/MyDrive/iran-prosperity-rag/data/index/chunks.json
Loaded 937 chunks from metadata

Setup complete!


## Setup Retriever

Configure the retriever with top_k results.

In [4]:
top_k = 5

# Create retriever from vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

print(f"Retriever configured with top_k={top_k}")

Retriever configured with top_k=5


## Test Retrieval

Test the retrieval system with a sample question.

In [5]:
query = "What are the main challenges discussed in the document?"

print(f"Query: {query}\n")
print("="*60)

# Retrieve documents using invoke (current LangChain API)
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents:\n")

for rank, doc in enumerate(docs, 1):
    # Find corresponding chunk metadata
    chunk_id = None
    page = None

    # Try to find matching chunk by text content
    for chunk in chunks_data:
        if chunk["text"] == doc.page_content:
            chunk_id = chunk["chunk_id"]
            page = chunk["page"]
            break

    # Extract score if available (FAISS doesn't always return scores directly)
    score = getattr(doc, 'metadata', {}).get('score', 'N/A')

    snippet = doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content

    print(f"Rank {rank}:")
    print(f"  Score: {score}")
    print(f"  Page: {page}")
    print(f"  Chunk ID: {chunk_id}")
    print(f"  Snippet: {snippet}")
    print()

Query: What are the main challenges discussed in the document?

Retrieved 5 documents:

Rank 1:
  Score: N/A
  Page: 29
  Chunk ID: 179
  Snippet: Mitigation  measures  for  political  challenges:  
●  Inclusive  dialogue,  power-sharing,  amnesty,  transparent  communication,  inﬂuencers  
●  Retention  incentives,  clear  mandates,  training,  whistleblower  protection,  gradual  transitions  
●  Interim  legal  frameworks,  independent  ove...

Rank 2:
  Score: N/A
  Page: 28
  Chunk ID: 178
  Snippet: Anticipated  Political  Challenges:  
●  Resistance  from  factions  opposing  the  transitional  government.  
●  Potential  lack  of  cooperation  from  existing  bureaucracies.  
●  Political  instability  affecting  resource  allocation  and  decision-making  processes.   
EMERGENCY  PHASE  |   ...

Rank 3:
  Score: N/A
  Page: 132
  Chunk ID: 736
  Snippet: regulations
 
or
 
pose
 
clear
 
conﬂicts
 
of
 
interest.
 
This
 
mechanism
 
should
 
be
 
empowered
 
with
 
legal
 
au

## LLM-Based Answer Generation

Implement LLM-based answer generation with guardrails and citations using OpenAI.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Debug toggle
DEBUG = False

def clean_text(text):
    """
    Normalize whitespace and newlines in text.

    Args:
        text: Input text string

    Returns:
        Cleaned text string
    """
    if not text:
        return ""
    # Replace multiple whitespace with single space
    import re
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

def answer_with_llm_and_citations(question, top_k=5):
    """
    Generate LLM-based answer from retrieved chunks with guardrails and citations.

    Args:
        question: User's question string
        top_k: Number of documents to retrieve (default: 5)

    Returns:
        Answer string with citations
    """
    # Check for API key
    if "OPENAI_API_KEY" not in os.environ:
        return "Error: OPENAI_API_KEY environment variable not set. Please set it before using the LLM."

    # Retrieve relevant documents
    docs = retriever.invoke(question)

    # Guardrail: if no docs, return refusal
    if not docs or len(docs) == 0:
        return "I don't know based on the provided documents."

    # Build context from retrieved chunks
    context_chunks = []
    pages_used = set()
    total_text_length = 0

    for doc in docs:
        # Find matching chunk metadata
        for chunk in chunks_data:
            if chunk["text"] == doc.page_content:
                cleaned_text = clean_text(chunk["text"])
                context_chunks.append({
                    "text": cleaned_text,
                    "page": chunk["page"],
                    "chunk_id": chunk["chunk_id"]
                })
                total_text_length += len(cleaned_text)
                if chunk["page"] is not None:
                    pages_used.add(chunk["page"])
                break

    # Guardrail: context sufficiency check
    if total_text_length < 800:
        return "I don't know based on the provided documents."

    if not context_chunks:
        return "I don't know based on the provided documents."

    # Debug output
    if DEBUG:
        print(f"Retrieved {len(context_chunks)} chunks from pages: {sorted(pages_used)}")
        for i, chunk in enumerate(context_chunks[:3], 1):
            snippet = chunk["text"][:200] + "..." if len(chunk["text"]) > 200 else chunk["text"]
            print(f"  Chunk {i} (p.{chunk['page']}): {snippet}")
        print()

    # Build context string
    context_parts = []
    for chunk in context_chunks:
        context_parts.append(chunk["text"])
    context = "\n\n".join(context_parts)
    context = clean_text(context)

    # Create citations string
    citations = ""
    if pages_used:
        sorted_pages = sorted(pages_used)
        citations = " ".join([f"[p.{p}]" for p in sorted_pages])

    # Initialize LLM
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

    # Build prompt with strict instructions
    system_prompt = """You are a helpful assistant that answers questions based ONLY on the provided context.

CRITICAL RULES:
1. Answer ONLY using information explicitly stated in the provided context.
2. If the answer is not in the context, say "I don't know based on the provided documents."
3. Do NOT invent facts, numbers, or sources.
4. Do NOT use information from outside the provided context.
5. Keep your answer concise: maximum 8 bullet points or approximately 150 words.
6. Be accurate and factual."""

    user_prompt = f"""Context from the document:

{context}

Question: {question}

Answer the question using ONLY the information from the context above. If the answer is not in the context, say you don't know."""

    # Generate answer
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]

    try:
        response = llm.invoke(messages)
        answer = response.content.strip()

        # Append citations
        if citations:
            answer += f"\n\nSources: {citations}"

        return answer
    except Exception as e:
        return f"Error generating answer: {str(e)}"

# Test the function (requires OPENAI_API_KEY to be set)
print("Note: Set OPENAI_API_KEY environment variable before testing.")
print("Example: os.environ['OPENAI_API_KEY'] = 'your-key-here'")
print()

Note: Set OPENAI_API_KEY environment variable before testing.
Example: os.environ['OPENAI_API_KEY'] = 'your-key-here'



## Interactive Chat Loop

Run this cell to start an interactive chat session. Type 'exit' or press Enter with empty input to stop.

In [7]:
# Set your OpenAI API key (if not already set)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("="*60)
print("RAG Chat System (LLM-based)")
print("Ask questions about the document. Type 'exit' or press Enter to quit.")
if "OPENAI_API_KEY" not in os.environ:
    print("WARNING: OPENAI_API_KEY not set. Please set it before using the chat.")
print("="*60)
print()

while True:
    question = input("You: ").strip()

    if not question or question.lower() == 'exit':
        print("\nGoodbye!")
        break

    print("\nAnswer:")
    answer = answer_with_llm_and_citations(question, top_k=top_k)
    print(answer)
    print("\n" + "-"*60 + "\n")

RAG Chat System (LLM-based)
Ask questions about the document. Type 'exit' or press Enter to quit.

You: What this plan is about?

Answer:
Error: OPENAI_API_KEY environment variable not set. Please set it before using the LLM.

------------------------------------------------------------



KeyboardInterrupt: Interrupted by user